In [3]:
import cv2
import numpy as np
import mediapipe as mp
from tensorflow.keras.models import load_model
from sklearn.preprocessing import LabelEncoder

# MediaPipe ile el tespiti
mp_hands = mp.solutions.hands
hand_tracker = mp_hands.Hands(min_detection_confidence=0.5, min_tracking_confidence=0.5)

# Modelinizi yükleyin (eğitilen model)
model = load_model('/Users/aksanurkonuk/PycharmProjects/Two-Way-Real-Time-Sign-Language-Converter/model/sign_language_model.h5')

# Etiket encoder'ı (etiketlerinizi burada belirtin)
label_encoder = LabelEncoder()
label_encoder.fit(['açıklamak', 'adres', 'akraba'])  # Eğitilen etiketlerinizi buraya ekleyin

# Video kaynağını başlatın
cap = cv2.VideoCapture(0)

# Hareket verisi saklama
prev_coordinates = None
motion_sequence = []
sequence_length = 90  # Bu, eğitimde kullandığınız dizin uzunluğuna eşit olmalıdır
frame_count = 0  # Karar verme sıklığı için bir sayaç
display_text = ""  # Ekranda gösterilecek metin
display_time = 0  # Yazının ne kadar süre gösterileceğini tutan sayaç

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # RGB formatına çevir
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hand_tracker.process(rgb_frame)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            curr_coordinates = [[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark]

            if prev_coordinates is not None:
                # Hareketi hesapla
                motion_vector = []
                for prev, curr in zip(prev_coordinates, curr_coordinates):
                    dx = curr[0] - prev[0]
                    dy = curr[1] - prev[1]
                    dz = curr[2] - prev[2]
                    motion_vector.append([dx, dy, dz])
                motion_sequence.append(motion_vector)

                # Veri uzunluğu tamamlanınca tahmin yap
                if len(motion_sequence) == sequence_length:
                    input_sequence = np.array(motion_sequence).reshape(1, sequence_length, 63)
                    pred = model.predict(input_sequence)
                    predicted_label = label_encoder.inverse_transform([np.argmax(pred)])

                    # Yazıyı ekranda 5 saniye boyunca göster
                    display_text = f"Predicted: {predicted_label[0]}"
                    display_time = 150  # 150 frame = yaklaşık 5 saniye

                    motion_sequence = []  # Diziyi sıfırla

            prev_coordinates = curr_coordinates

    # Yazıyı ekranda tutmak
    if display_time > 0:
        display_time -= 1  # Zamanlayıcıyı azalt
        cv2.putText(frame, display_text, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 3, cv2.LINE_AA)

    # Sonuçları ekranda göster
    cv2.imshow("Sign Language Detection", frame)

    # 'q' tuşuna basılınca çık
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


I0000 00:00:1734120100.654180 1223694 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 88.1), renderer: Apple M1
W0000 00:00:1734120100.688922 1227520 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1734120100.702702 1227519 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
